In [0]:
dbutils.library.restartPython()

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # ETL Pipeline — Run on Databricks
# MAGIC Reads the raw telemetry CSV your data-generation notebook wrote to ADLS Gen2,
# MAGIC runs the same cleaning + feature-engineering logic as `src/etl_pipeline.py`
# MAGIC (null imputation, rolling risk features), and writes the result as a Delta
# MAGIC table partitioned by region.
# MAGIC
# MAGIC **Before running this:** storage OAuth auth must be set at the CLUSTER level
# MAGIC (Compute → your cluster → Edit → Advanced Options → Spark → Spark Config),
# MAGIC not inline here — see the earlier setup notes. If this notebook errors with
# MAGIC `CONFIG_NOT_AVAILABLE`, you're on Serverless compute; switch the compute
# MAGIC selector (top-right) to your named classic cluster instead.

# COMMAND ----------

storage_account = "enterprisemlopssa"  # <-- change if yours differs

raw_path = f"abfss://raw@{storage_account}.dfs.core.windows.net/asset_telemetry.csv"
processed_path = f"abfss://processed@{storage_account}.dfs.core.windows.net/asset_features"

# COMMAND ----------

# MAGIC %md
# MAGIC ## Import the ETL logic straight from the repo
# MAGIC This repo is checked out via Databricks Repos, so `src/etl_pipeline.py` is
# MAGIC already on disk in the workspace — we just need it on `sys.path`.

# COMMAND ----------

import os
import sys

# os.getcwd() returns the folder containing THIS notebook (DBR 11.0+).
# Adjust the ".." count if you move this notebook to a different depth.
notebook_dir = os.getcwd()
repo_root = os.path.abspath(os.path.join(notebook_dir, ".."))
src_dir = os.path.join(repo_root, "src")
sys.path.insert(0, src_dir)

print(f"Repo root resolved to: {repo_root}")
print(f"Looking for etl_pipeline.py in: {src_dir}")

from etl_pipeline import run_spark_pipeline # noqa: E402
from etl_pipeline import NUMERIC_SENSOR_COLS, ROLLING_WINDOW  # noqa: E402
from pyspark.sql import Window
from pyspark.sql import functions as F

tenant_id = dbutils.secrets.get(scope="storage-auth", key="tenant-id")
azure_storage_options = {
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net": dbutils.secrets.get(scope="storage-auth", key="client-id"),
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net": dbutils.secrets.get(scope="storage-auth", key="client-secret"),
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
}



# COMMAND ----------

# MAGIC %md
# MAGIC ## Run the pipeline
# MAGIC On Databricks, `run_spark_pipeline` reuses this notebook's already-active
# MAGIC Spark session (Databricks Runtime bundles Delta Lake natively, so the
# MAGIC `configure_spark_with_delta_pip` fallback in the script quietly no-ops here).

# COMMAND ----------

run_spark_pipeline(raw_path, processed_path)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Quick sanity check on the output

# COMMAND ----------

#result_df = spark.read.options(**azure_storage_options).format("delta").load(processed_path)
result_df = spark.read.format("delta").load(processed_path)
print(f"Row count: {result_df.count():,}")
display(result_df.limit(10))

# COMMAND ----------

# MAGIC %md
# MAGIC ## Check the risk-flag distribution
# MAGIC A quick eyeball check that the feature engineering produced a sensible
# MAGIC (non-degenerate) split, before moving on to the training notebook.

# COMMAND ----------

result_df.groupBy("risk_flag").count().show()